# 02 - PySpark Batch NLP Inference (3-class sentiment)

Continues from `01_spark_intro.ipynb`, now with the **retrained 3-class model**.

| label | meaning | original stars |
|---|---|---|
| 0 | negative | 1-2 |
| 1 | neutral | 3 |
| 2 | positive | 4-5 |

```text
Books_rating.csv -> Spark cleaning -> clean Parquet -> 3-class DistilBERT UDF -> evaluation -> predictions Parquet
```

**Reference to beat / match (Colab test set, 5,000 rows):** accuracy 0.84, macro-F1 0.67, neutral F1 0.38.

**Before running:** unzip `best_model.zip` into `project/model/best_model/` (it must contain `config.json`, `model.safetensors`, and the tokenizer files).

## 0. Imports, Spark session and configuration

In [1]:
from pathlib import Path
from typing import Iterator
import time

import pandas as pd
import torch

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lower, regexp_replace, trim, length, when, count, avg, pandas_udf,
)
from pyspark.sql.types import IntegerType
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

spark = (
    SparkSession.builder
    .appName("NLP-Batch-Inference-3class")
    .master("local[4]")
    .config("spark.driver.memory", "5g")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
# Absolute paths: Spark Python workers may start in a different working directory,
# so relative paths can fail there.
DATA_PATH   = str(Path("../data/raw/Books_rating.csv").resolve())
CLEAN_PATH  = str(Path("../data/processed/books_reviews_clean").resolve())
OUTPUT_PATH = str(Path("../data/processed/books_reviews_predictions_3class").resolve())
MODEL_PATH  = str(Path("../model/base_model").resolve())   # the NEW 3-class model

ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}
LABEL_NAMES = [ID2LABEL[i] for i in range(3)]

# Inference settings
MAX_LENGTH    = 128   # training used 128; 64 is faster on CPU -> verify in section 4
BATCH_SIZE    = 16
TORCH_THREADS = 1     # threads per Spark worker (with local[4] -> 4 workers x 1 thread)

# Run sizes (CPU: expect roughly 10-20 reviews/sec in total)
N_VALIDATION      = 5_000
NUM_PARTITIONS    = 8
FULL_RUN_FRACTION = 0.005   # ~15,000 reviews; raise it if you have time or a GPU
SEED = 42

## 1. Load the raw CSV

`inferSchema` is off on purpose: it forces an extra full pass over 3M rows, and we cast the columns we need explicitly anyway.

In [3]:
df = spark.read.csv(DATA_PATH, header=True)
df.printSchema()

root
 |-- Id: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- User_id: string (nullable = true)
 |-- profileName: string (nullable = true)
 |-- review/helpfulness: string (nullable = true)
 |-- review/score: string (nullable = true)
 |-- review/time: string (nullable = true)
 |-- review/summary: string (nullable = true)
 |-- review/text: string (nullable = true)



## 2. Clean, label, and save as Parquet

Cleaning runs **once** and its result is saved to Parquet. Everything after this reads Parquet (fast, columnar) instead of re-scanning the CSV.

In [4]:
# Cast numeric columns
df = (
    df
    .withColumn("review/score", col("review/score").cast("float"))
    .withColumn("review/time", col("review/time").cast("long"))
)

# Keep valid 1-5 ratings and non-null text
df = (
    df
    .filter(col("review/score").isNotNull())
    .filter((col("review/score") >= 1) & (col("review/score") <= 5))
    .filter(col("review/text").isNotNull())
)

# Text normalization: lowercase (harmless only for an uncased model - check your tokenizer),
# strip HTML tags and entities, collapse whitespace, drop empty reviews
df = (
    df
    .withColumn("review/text", lower(col("review/text")))
    .withColumn("review/text", regexp_replace(col("review/text"), r"<[^>]+>", " "))
    .withColumn("review/text", regexp_replace(col("review/text"), r"&[a-zA-Z0-9#]+;", " "))
    .withColumn("review/text", regexp_replace(col("review/text"), r"\s+", " "))
    .withColumn("review/text", trim(col("review/text")))
    .filter(length(col("review/text")) > 0)
)

# Remove exact duplicate reviews
df = df.dropDuplicates(["Id", "User_id", "review/time", "review/text"])

# 3-class label (Spark syntax, NOT pandas): 1-2 -> 0, 3 -> 1, 4-5 -> 2
df = df.withColumn(
    "label",
    when(col("review/score") <= 2, 0)
    .when(col("review/score") == 3, 1)
    .otherwise(2),
)

clean_df = df.select(
    "Id", "Title", "User_id", "review/score", "review/time",
    "review/summary", "review/text", "label",
)

In [5]:
start = time.perf_counter()
clean_df.write.mode("overwrite").parquet(CLEAN_PATH)
print(f"Clean Parquet written in {(time.perf_counter() - start) / 60:.1f} min -> {CLEAN_PATH}")

Clean Parquet written in 8.6 min -> /home/jovyan/work/data/processed/books_reviews_clean


In [6]:
clean_df = spark.read.parquet(CLEAN_PATH)
n_clean = clean_df.count()
print(f"Clean rows: {n_clean:,}")
print("Partitions:", clean_df.rdd.getNumPartitions())

print("Class distribution (full clean data):")
(
    clean_df.groupBy("label").count().orderBy("label")
    .withColumn("share", col("count") / n_clean)
    .show()
)

Clean rows: 2,958,041
Partitions: 16
Class distribution (full clean data):
+-----+-------+-------------------+
|label|  count|              share|
+-----+-------+-------------------+
|    0| 349422|0.11812615173352904|
|    1| 251200|0.08492106769311177|
|    2|2357419| 0.7969527805733592|
+-----+-------+-------------------+



## 3. Inference UDF (model loaded once per task)

An **iterator-style pandas UDF** loads the tokenizer and model once at the start of each task, then processes every batch of that task. This avoids reloading the 250 MB model for each batch, and it doesn't depend on global variables surviving between tasks.

The model is dynamically quantized (int8) for faster CPU inference. Remove that line if you run on GPU.

In [7]:
def load_model():
    torch.set_num_threads(TORCH_THREADS)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    model.eval()
    model = torch.quantization.quantize_dynamic(
        model, {torch.nn.Linear}, dtype=torch.qint8
    )
    return tokenizer, model


def predict_batch(texts, tokenizer, model):
    predictions = []
    with torch.no_grad():
        for start in range(0, len(texts), BATCH_SIZE):
            chunk = texts[start:start + BATCH_SIZE]
            inputs = tokenizer(
                chunk,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            logits = model(**inputs).logits
            predictions.extend(torch.argmax(logits, dim=1).tolist())
    return predictions


@pandas_udf(IntegerType())
def predict_label(batches: Iterator[pd.Series]) -> Iterator[pd.Series]:
    tokenizer, model = load_model()          # once per task
    for texts in batches:
        preds = predict_batch(texts.tolist(), tokenizer, model)
        yield pd.Series(preds, dtype="int32")

## 4. Validation on a random sample

A random sample (not `.limit()` on the first rows, which come from only a few books). It only has to show that the distributed UDF works and that the metrics match the Colab test report.

In [8]:
val_fraction = min(1.0, 3 * N_VALIDATION / n_clean)

start = time.perf_counter()
val_df = (
    clean_df
    .sample(withReplacement=False, fraction=val_fraction, seed=SEED)
    .limit(N_VALIDATION)
    .select("Id", "label", "review/text")
    .repartition(4)
    .withColumn("predicted_label", predict_label(col("review/text")))
    .cache()
)
n_val = val_df.count()
elapsed = time.perf_counter() - start
print(f"Rows: {n_val:,} | {elapsed:.1f} s | {n_val / elapsed:.1f} reviews/sec")

Rows: 5,000 | 420.4 s | 11.9 reviews/sec


In [9]:
eval_val = val_df.select("label", "predicted_label").toPandas()   # 2 small columns only

print(classification_report(
    eval_val["label"], eval_val["predicted_label"],
    target_names=LABEL_NAMES, digits=3,
))
print("Macro-F1:", round(f1_score(eval_val["label"], eval_val["predicted_label"], average="macro"), 4))
print("Confusion matrix (rows = true, cols = predicted):")
print(confusion_matrix(eval_val["label"], eval_val["predicted_label"]))

              precision    recall  f1-score   support

    negative      0.697     0.660     0.678       623
     neutral      0.311     0.332     0.321       425
    positive      0.924     0.925     0.925      3952

    accuracy                          0.842      5000
   macro avg      0.644     0.639     0.641      5000
weighted avg      0.844     0.842     0.843      5000

Macro-F1: 0.6411
Confusion matrix (rows = true, cols = predicted):
[[ 411  108  104]
 [  89  141  195]
 [  90  205 3657]]


**Checkpoint:** macro-F1 should be close to 0.67 (neutral clearly the weakest class). If it is much lower, check `MODEL_PATH` (must be the 3-class model), the label mapping, and try `MAX_LENGTH = 128`.

## 5. Batch inference on a larger sample, saved to Parquet

No `.cache()` and no `toPandas()` on the full result: Spark computes the predictions while writing Parquet. At about 14 reviews/sec, the full 3M rows would need days on CPU, so this run uses a sample (`FULL_RUN_FRACTION`).

In [10]:
start = time.perf_counter()

predictions_df = (
    clean_df
    .sample(withReplacement=False, fraction=FULL_RUN_FRACTION, seed=SEED)
    .repartition(NUM_PARTITIONS)
    .withColumn("predicted_label", predict_label(col("review/text")))
    .withColumn(
        "predicted_name",
        when(col("predicted_label") == 0, "negative")
        .when(col("predicted_label") == 1, "neutral")
        .otherwise("positive"),
    )
)

predictions_df.write.mode("overwrite").parquet(OUTPUT_PATH)

elapsed = (time.perf_counter() - start) / 60
print(f"Done in {elapsed:.1f} min -> {OUTPUT_PATH}")

Done in 15.0 min -> /home/jovyan/work/data/processed/books_reviews_predictions_3class


## 6. Spark-native evaluation on the saved predictions

In [11]:
saved_df = spark.read.parquet(OUTPUT_PATH)
n_saved = saved_df.count()
print(f"Saved rows: {n_saved:,}")
saved_df.printSchema()

print("Actual label distribution:")
saved_df.groupBy("label").count().orderBy("label").show()

print("Predicted label distribution:")
saved_df.groupBy("predicted_label").count().orderBy("predicted_label").show()

print("Actual vs predicted:")
saved_df.groupBy("label", "predicted_label").count().orderBy("label", "predicted_label").show()

accuracy = saved_df.select(
    avg(when(col("label") == col("predicted_label"), 1.0).otherwise(0.0)).alias("accuracy")
)
accuracy.show()

Saved rows: 14,811
root
 |-- Id: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- User_id: string (nullable = true)
 |-- review/score: float (nullable = true)
 |-- review/time: long (nullable = true)
 |-- review/summary: string (nullable = true)
 |-- review/text: string (nullable = true)
 |-- label: integer (nullable = true)
 |-- predicted_label: integer (nullable = true)
 |-- predicted_name: string (nullable = true)

Actual label distribution:
+-----+-----+
|label|count|
+-----+-----+
|    0| 1788|
|    1| 1246|
|    2|11777|
+-----+-----+

Predicted label distribution:
+---------------+-----+
|predicted_label|count|
+---------------+-----+
|              0| 1643|
|              1| 1298|
|              2|11870|
+---------------+-----+

Actual vs predicted:
+-----+---------------+-----+
|label|predicted_label|count|
+-----+---------------+-----+
|    0|              0| 1162|
|    0|              1|  270|
|    0|              2|  356|
|    1|              0|  237|
|   

In [12]:
# Per-class report: only two integer columns are collected
eval_saved = saved_df.select("label", "predicted_label").toPandas()
print(classification_report(
    eval_saved["label"], eval_saved["predicted_label"],
    target_names=LABEL_NAMES, digits=3,
))
print("Macro-F1:", round(f1_score(eval_saved["label"], eval_saved["predicted_label"], average="macro"), 4))

              precision    recall  f1-score   support

    negative      0.707     0.650     0.677      1788
     neutral      0.347     0.361     0.354      1246
    positive      0.923     0.930     0.927     11777

    accuracy                          0.848     14811
   macro avg      0.659     0.647     0.653     14811
weighted avg      0.848     0.848     0.848     14811

Macro-F1: 0.6526


## 7. Sanity checks

In [13]:
null_preds = saved_df.filter(col("predicted_label").isNull()).count()
out_of_range = saved_df.filter(
    (col("predicted_label") < 0) | (col("predicted_label") > 2)
).count()
null_text = saved_df.filter(col("review/text").isNull()).count()

print("Null predictions:        ", null_preds)
print("Predictions outside 0-2: ", out_of_range)
print("Null review text:        ", null_text)
assert null_preds == 0 and out_of_range == 0 and null_text == 0, "Sanity check failed"
print("All checks passed.")

saved_df.select("label", "predicted_name", "review/text").show(10, truncate=100)

Null predictions:         0
Predictions outside 0-2:  0
Null review text:         0
All checks passed.
+-----+--------------+----------------------------------------------------------------------------------------------------+
|label|predicted_name|                                                                                         review/text|
+-----+--------------+----------------------------------------------------------------------------------------------------+
|    1|      negative|this review is for the mp3 cd unabridged version of dragonflight put out by blilliance audio.firs...|
|    2|      positive|this was my first david morrell book. i heard of him as i researched books that were nominated fo...|
|    2|      positive|i purchased this along with women without doctors . this book did appear to have been used slight...|
|    2|      positive|the author writes in a very refined style, making it easy for anyone to grasp the concepts. even ...|
|    2|      positive|we have

## Phase 3 checkpoint

- Spark ingestion and cleaning (deduplicated, labeled) -> clean Parquet
- 3-class DistilBERT inference with an iterator pandas UDF
- Validation against the Colab test metrics
- Sample batch inference saved to Parquet, evaluated in Spark

**Next: Phase 4 - Kafka -> Spark Structured Streaming -> the same `predict_label` logic.** The model is the bottleneck on CPU (about 14 reviews/sec), so streaming will need small batches per trigger, a GPU, or both.